# Multi-Agent Research Lab — Demo Notebook

Notebook này demo toàn bộ pipeline:
1. Single-agent baseline
2. Multi-agent workflow (Supervisor → Researcher → Analyst → Writer → Critic)
3. Benchmark comparison
4. Trace inspection

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('../src').resolve()))

from multi_agent_research_lab.observability.logging import configure_logging
configure_logging('WARNING')  # quiet logs in notebook

## 1. Single-Agent Baseline

In [ ]:
from time import perf_counter
from multi_agent_research_lab.services.llm_client import LLMClient

QUERY = "Explain GraphRAG and its advantages over traditional RAG"

llm = LLMClient()
t0 = perf_counter()
resp = llm.complete(
    "You are a research assistant. Answer comprehensively in ~500 words.",
    QUERY
)
baseline_latency = perf_counter() - t0

print(f"Latency : {baseline_latency:.2f}s")
print(f"Tokens  : {resp.input_tokens}in / {resp.output_tokens}out")
print(f"Cost    : ${resp.cost_usd:.5f}" if resp.cost_usd else "Cost: n/a (mock)")
print()
print(resp.content[:500], "...")

## 2. Multi-Agent Workflow

In [ ]:
from multi_agent_research_lab.core.schemas import ResearchQuery
from multi_agent_research_lab.core.state import ResearchState
from multi_agent_research_lab.graph.workflow import MultiAgentWorkflow

state = ResearchState(request=ResearchQuery(query=QUERY))

t0 = perf_counter()
result = MultiAgentWorkflow().run(state)
multi_latency = perf_counter() - t0

print(f"Latency      : {multi_latency:.2f}s")
print(f"Iterations   : {result.iteration}")
print(f"Route history: {result.route_history}")
print(f"Errors       : {result.errors or 'none'}")
print()
print(result.final_answer[:500], "...")

## 3. Agent Results (tokens + cost per agent)

In [ ]:
import pandas as pd

rows = [
    {
        "agent": r.agent,
        "output_tokens": r.metadata.get("output_tokens"),
        "cost_usd": r.metadata.get("cost_usd"),
        "quality_score": r.metadata.get("quality_score"),
    }
    for r in result.agent_results
]
pd.DataFrame(rows)

## 4. Benchmark Comparison

In [ ]:
from multi_agent_research_lab.evaluation.benchmark import run_comparison

baseline_m, multi_m = run_comparison(QUERY)

data = {
    "Run": [baseline_m.run_name, multi_m.run_name],
    "Latency (s)": [baseline_m.latency_seconds, multi_m.latency_seconds],
    "Cost (USD)": [baseline_m.estimated_cost_usd, multi_m.estimated_cost_usd],
    "Quality": [baseline_m.quality_score, multi_m.quality_score],
    "Notes": [baseline_m.notes, multi_m.notes],
}
pd.DataFrame(data)

## 5. Trace Inspection

In [ ]:
import json
from multi_agent_research_lab.observability.tracing import save_trace_to_file

trace_path = save_trace_to_file(result, directory='../reports/traces')
print(f"Trace saved → {trace_path}")

with open(trace_path) as f:
    trace = json.load(f)

print(json.dumps(trace['route_history'], indent=2))
print()
for event in trace['trace_events']:
    print(event['name'], event['payload'])